In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [5]:
train_path = r'../data/03_train.csv'
df_train = pd.read_csv(train_path)

print(f"\n=== Train data information ===")
print(f"Total rows: {len(df_train):,}")
print(f"Total columns: {len(df_train.columns)}")

print(f"\nLabel distribution:")
label_counts = df_train['Label'].value_counts()
for label, count in label_counts.items():
    pct = (count / len(df_train)) * 100
    print(f"  {label:30s}: {count:5,d} samples")

# Seperate Benign data
benign = df_train[df_train['Label'] == 'Benign'].copy()
print(f"\n=== Benign 데이터 ===")
print(f"Benign samples: {len(benign):,} ({len(benign)/len(df_train)*100:.2f}%)")


# Select only numerical features
# Exclude Timestamp and Label
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n=== Features for Analysis ===")
print(f"Number of numerical features: {len(numeric_cols)}")
print(f"(Timestamp, Label excluded)")


# Extract attack labels
attack_labels = [label for label in df_train['Label'].unique() if label != 'Benign']
attack_labels.sort()  # 알파벳 순서

print(f"\n=== Attack types to Analyze ===")
print(f"Total attacks: {len(attack_labels)}")
for i, attack in enumerate(attack_labels, 1):
    count = (df_train['Label'] == attack).sum()
    print(f"  {i:2d}. {attack:30s} {count:,} samples ")



=== Train data information ===
Total rows: 23,846
Total columns: 80

Label distribution:
  Benign                        : 11,307 samples
  DoS attacks-GoldenEye         : 2,363 samples
  DDoS attacks-LOIC-HTTP        : 2,349 samples
  FTP-BruteForce                : 2,324 samples
  DoS attacks-SlowHTTPTest      : 2,321 samples
  Bot                           : 1,327 samples
  DDOS attack-LOIC-UDP          : 1,112 samples
  Brute Force -Web              :   489 samples
  Brute Force -XSS              :   184 samples
  SQL Injection                 :    70 samples

=== Benign 데이터 ===
Benign samples: 11,307 (47.42%)

=== Features for Analysis ===
Number of numerical features: 78
(Timestamp, Label excluded)

=== Attack types to Analyze ===
Total attacks: 9
   1. Bot                            1,327 samples 
   2. Brute Force -Web               489 samples 
   3. Brute Force -XSS               184 samples 
   4. DDOS attack-LOIC-UDP           1,112 samples 
   5. DDoS attacks-LOIC-HTTP   

In [10]:
# Feature analysis function for each attack
def analyze_attack_features(df_train, benign, attack_label, numeric_cols, alpha=0.001):
    """
    Analyze all features for a specific attack
    
    Returns:
        DataFrame with columns: feature, benign_mean, attack_mean, ratio, 
                               t_stat, p_value, cohens_d, significant
    """
    # Extract attack data
    attack = df_train[df_train['Label'] == attack_label].copy()
    
    if len(attack) == 0:
        print(f"Warning: {attack_label} has no samples")
        return pd.DataFrame()
    
    results = []
    
    for feat in numeric_cols:
        # Extract data and remove missing values
        benign_vals = benign[feat].dropna()
        attack_vals = attack[feat].dropna()
        
        # If no data available, skip
        if len(benign_vals) == 0 or len(attack_vals) == 0:
            continue
        
        # Calculate statistics
        benign_mean = benign_vals.mean()
        benign_std = benign_vals.std()
        attack_mean = attack_vals.mean()
        attack_std = attack_vals.std()
        
        # Calculate ratio 
        ratio = attack_mean / benign_mean if benign_mean != 0 else np.nan
        
        # T-test
        try:
            t_stat, p_val = stats.ttest_ind(benign_vals, attack_vals, equal_var=False)
        except:
            t_stat, p_val = np.nan, 1.0
        
        # Effect Size (Cohen's d)
        pooled_std = np.sqrt((benign_std**2 + attack_std**2) / 2)
        cohens_d = abs(attack_mean - benign_mean) / pooled_std if pooled_std != 0 else 0
        
        results.append({
            'feature': feat,
            'benign_mean': benign_mean,
            'benign_std': benign_std,
            'attack_mean': attack_mean,
            'attack_std': attack_std,
            'ratio': ratio,
            't_stat': t_stat,
            'p_value': p_val,
            'cohens_d': cohens_d,
            'significant': p_val < alpha
        })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df


## Attack Analysis

- **attack samples**: Number of samples for the given attack in the training set  
- **total features analyzed**: Number of numerical features analyzed  
- **significant (p < 0.001)**: Number of features where the distribution between benign and attack is statistically different  

→ **p-value**: Probability that the null hypothesis (feature values of benign and attack are the same) is true  
→ **p < 0.001**: The probability that benign and attack are actually the same but appear different by chance is less than 0.1% (i.e., 99.9% confidence)

---

### Top 10 Features

Top 10 features with the largest effect size (Cohen’s d) among statistically significant features  
→ Features that clearly distinguish between benign and attack traffic

- Benign / Attack: Mean values  
- ratio: Attack mean / Benign mean  
- Effect: Magnitude of difference between the two distributions  

+ Larger ratio and effect size indicate features more useful for detection

In [11]:
# Analyze all attacks
print("Starting feature analysis for each attack")

all_results = {}

for attack in attack_labels:
    print(f"\n\n[{attack}]")
    
    # Run analysis
    results = analyze_attack_features(df_train, benign, attack, numeric_cols)
    
    if len(results) == 0:
        print("No results (no samples or all features invalid)")
        continue
    
    # Select only significant features
    significant = results[results['significant']].copy()
    
    # Sort by effect size
    significant = significant.sort_values('cohens_d', ascending=False)
    
    # Store results
    all_results[attack] = {
        'all': results,
        'significant': significant
    }
    
    # Print summary
    attack_count = (df_train['Label'] == attack).sum()
    print(f"Attack samples: {attack_count:,}")
    print(f"Total features analyzed: {len(results)}")
    print(f"Significant features (p<0.001): {len(significant)}")
    
    # Print Top 10
    print(f"\n{'Rank':>4s} {'Feature':30s} {'Benign':>12s} {'Attack':>12s} {'Ratio':>8s} {'Effect':>8s}")
    print("-" * 80)
    
    for idx in range(min(10, len(significant))):
        row = significant.iloc[idx]
        rank = idx + 1
        feat = row['feature']
        b_mean = row['benign_mean']
        a_mean = row['attack_mean']
        ratio = row['ratio']
        cohens = row['cohens_d']
        
        print(f"{rank:4d} {feat:30s} {b_mean:>12.2f} {a_mean:>12.2f} {ratio:>8.2f}x {cohens:>8.2f}")

Starting feature analysis for each attack


[Bot]
Attack samples: 1,327
Total features analyzed: 78
Significant features (p<0.001): 69

Rank Feature                              Benign       Attack    Ratio   Effect
--------------------------------------------------------------------------------
   1 Flow Duration                   48396759.32      5584.52     0.00x     1.36
   2 Fwd IAT Tot                     47962966.99       521.28     0.00x     1.34
   3 Flow IAT Max                    18994592.01      5115.23     0.00x     1.22
   4 Idle Max                        18651632.98         0.00     0.00x     1.20
   5 Fwd IAT Max                     18650146.34       494.45     0.00x     1.19
   6 Idle Mean                       13505644.01         0.00     0.00x     1.03
   7 Flow IAT Std                     5219834.85      1954.01     0.00x     1.03
   8 Fwd IAT Std                      5412677.00       159.91     0.00x     1.01
   9 Fwd IAT Mean                     8978279.58       

### Bot
- A bot is an individual software program that infects devices and enables them to automatically perform malicious activities under the control of an attacker, forming part of a botnet.
- Most high-level features are significantly smaller compared to benign traffic.

In [12]:
# Summary of results
print("Analysis Summary")

summary = []
for attack, data in all_results.items():
    sig_count = len(data['significant'])
    if sig_count > 0:
        top_feature = data['significant'].iloc[0]['feature']
        top_effect = data['significant'].iloc[0]['cohens_d']
    else:
        top_feature = "N/A"
        top_effect = 0.0
    
    summary.append({
        'attack': attack,
        'significant_features': sig_count,
        'top_feature': top_feature,
        'top_effect_size': top_effect
    })

summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values('significant_features', ascending=False)

print(f"\n{'Attack':30s} {'Sig Features':>15s} {'Top Feature':30s} {'Effect Size':>12s}")
print("-" * 90)
for _, row in summary_df.iterrows():
    print(f"{row['attack']:30s} {row['significant_features']:>15d} "
          f"{row['top_feature']:30s} {row['top_effect_size']:>12.2f}")

Analysis Summary

Attack                            Sig Features Top Feature                     Effect Size
------------------------------------------------------------------------------------------
DDoS attacks-LOIC-HTTP                      70 ECE Flag Cnt                           2.72
FTP-BruteForce                              70 Fwd Seg Size Min                       4.30
DDOS attack-LOIC-UDP                        70 Fwd Act Data Pkts                      7.30
DoS attacks-SlowHTTPTest                    70 Fwd Seg Size Min                       4.30
Bot                                         69 Flow Duration                          1.36
DoS attacks-GoldenEye                       68 Fwd Seg Size Min                       2.90
Brute Force -XSS                            65 TotLen Fwd Pkts                        1.28
Brute Force -Web                            62 TotLen Fwd Pkts                        0.73
SQL Injection                               56 Fwd IAT Tot              

In [13]:
# Save results

output_dir = r'../data/04'

# 1. Save all results into a single CSV file
all_results_list = []
for attack, data in all_results.items():
    df_temp = data['significant'].copy()
    df_temp['attack'] = attack
    all_results_list.append(df_temp)

if len(all_results_list) > 0:
    final_df = pd.concat(all_results_list, ignore_index=True)
    
    # Reorder columns
    cols = ['attack', 'feature', 'benign_mean', 'attack_mean', 'ratio', 
            'cohens_d', 'p_value', 't_stat', 'benign_std', 'attack_std']
    final_df = final_df[cols]
    
    output_path = f'{output_dir}/04_attack_features_analysis.csv'
    final_df.to_csv(output_path, index=False)
    print(f"\nSaved full results to: {output_path}")

# 2. Save Top 10 features per attack
top10_summary = []
for attack, data in all_results.items():
    top10 = data['significant'].head(10)
    for idx, row in top10.iterrows():
        top10_summary.append({
            'attack': attack,
            'rank': top10.index.get_loc(idx) + 1,
            'feature': row['feature'],
            'benign_mean': row['benign_mean'],
            'attack_mean': row['attack_mean'],
            'ratio': row['ratio'],
            'effect_size': row['cohens_d'],
            'p_value': row['p_value']
        })

if len(top10_summary) > 0:
    top10_df = pd.DataFrame(top10_summary)
    top10_path = f'{output_dir}/04_attack_top10_features.csv'
    top10_df.to_csv(top10_path, index=False)
    print(f"Saved Top 10 summary to: {top10_path}")

# 3. Save summary statistics
summary_path = f'{output_dir}/04_attack_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary statistics to: {summary_path}")

print("\n\nAll analysis completed!")
print(f"\n\nNext Steps:")
print(f"  1. Review the generated CSV files")
print(f"  2. Design detection rules based on top features")
print(f"  3. Validate rules using test data")


Saved full results to: ../data/04/04_attack_features_analysis.csv
Saved Top 10 summary to: ../data/04/04_attack_top10_features.csv
Saved summary statistics to: ../data/04/04_attack_summary.csv


All analysis completed!


Next Steps:
  1. Review the generated CSV files
  2. Design detection rules based on top features
  3. Validate rules using test data
